In [ ]:
%pip install boto3 pandas pyarrow

In [ ]:
import io
import json
import hashlib
from pathlib import Path

import boto3
import pandas as pd

BUCKET = "s3-stock-market-project-ashwin"

PREFIX = (
    "features_parquet/"
    "feature_version=close_features_v1/"
    "data_version=kaggle_2026_02/"
)

# Fetch one lookback snapshot (Jan 1) so the first requested panel week
# (the second Wednesday of January) has valid *_train values.
FETCH_START_WEEK = pd.Timestamp("2025-01-01")
PANEL_START_WEEK = pd.Timestamp("2025-01-08")
FETCH_END_DATE = pd.Timestamp("2025-02-28")

requested_weeks = pd.date_range(
    FETCH_START_WEEK, FETCH_END_DATE, freq="W-WED"
)

# Different bucket/prefix selections get different local cache files.
cache_id = hashlib.sha256(
    (
        f"{BUCKET}/{PREFIX}"
        f"{FETCH_START_WEEK.date()}/{FETCH_END_DATE.date()}"
    ).encode()
).hexdigest()[:12]

cache_path = Path("local_cache") / f"features_{cache_id}.parquet"

# False: use local cache when available, with zero S3 requests.
# True: reread the selected S3 files and replace the local cache.
REFRESH = False

if cache_path.exists() and not REFRESH:
    weekly_panel = pd.read_parquet(cache_path)
    print(f"Loaded local cache: {cache_path}")

else:
    session = boto3.Session(
        profile_name="admin",
        region_name="ap-southeast-2",
    )
    s3 = session.client("s3")

    # Pagination ensures we retrieve more than 1,000 objects if needed.
    keys = []
    paginator = s3.get_paginator("list_objects_v2")

    # Query only the required weekly partitions instead of listing the
    # entire historical prefix.
    for week in requested_weeks:
        week_prefix = f"{PREFIX}week_date={week.date()}/"
        for page in paginator.paginate(
            Bucket=BUCKET,
            Prefix=week_prefix,
        ):
            keys.extend(
                obj["Key"]
                for obj in page.get("Contents", [])
                if obj["Key"].endswith((".parquet", ".json"))
            )

    if not keys:
        raise ValueError(
            "No JSON/Parquet files found for the requested Jan-Feb "
            f"2025 partitions under s3://{BUCKET}/{PREFIX}"
        )

    print(f"Reading {len(keys):,} files from S3...")
    frames = []

    for count, key in enumerate(sorted(keys), start=1):
        body = s3.get_object(Bucket=BUCKET, Key=key)["Body"]
        try:
            content = body.read()
        finally:
            body.close()

        if key.endswith(".parquet"):
            frame = pd.read_parquet(io.BytesIO(content))
        else:
            # Handles the single-record JSON files from your Kafka consumer.
            records = json.loads(content.decode("utf-8"))
            frame = pd.DataFrame(
                records if isinstance(records, list) else [records]
            )

        # Recover partition columns when absent from the file itself.
        for component in key.split("/")[:-1]:
            if "=" in component:
                name, value = component.split("=", 1)
                if name not in frame.columns:
                    frame[name] = value

        required = {"Ticker", "week_date", "price_date", "adj_close"}
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"{key}: missing columns {sorted(missing)}")

        frames.append(frame)

        if count % 100 == 0 or count == len(keys):
            print(f"Read {count:,}/{len(keys):,} files")

    weekly_panel = pd.concat(frames, ignore_index=True)

    for column in ["week_date", "price_date"]:
        weekly_panel[column] = pd.to_datetime(
            weekly_panel[column], errors="raise"
        ).dt.normalize()

    # Do not silently combine duplicated JSON and Parquet snapshots.
    duplicates = weekly_panel.duplicated(
        ["Ticker", "week_date"], keep=False
    )

    if duplicates.any():
        display(
            weekly_panel.loc[
                duplicates, ["Ticker", "week_date"]
            ].head(20)
        )
        raise ValueError(
            "Duplicate stock/week rows found. Check for overlapping "
            "JSON and Parquet files before proceeding."
        )

    weekly_panel = (
        weekly_panel
        .sort_values(["week_date", "Ticker"])
        .reset_index(drop=True)
    )

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    weekly_panel.to_parquet(cache_path, index=False)

    print(f"Saved local cache: {cache_path}")

# Normalize cached and freshly downloaded data identically, then verify
# that every requested Wednesday partition was actually present.
for column in ["week_date", "price_date"]:
    weekly_panel[column] = pd.to_datetime(
        weekly_panel[column], errors="raise"
    ).dt.normalize()

weekly_panel = weekly_panel.loc[
    weekly_panel["week_date"].between(
        FETCH_START_WEEK, FETCH_END_DATE
    )
].copy()
available_weeks = pd.DatetimeIndex(
    weekly_panel["week_date"].drop_duplicates().sort_values()
)
missing_weeks = requested_weeks.difference(available_weeks)
if len(missing_weeks):
    raise ValueError(
        "Missing requested S3 week partitions: "
        f"{missing_weeks.strftime('%Y-%m-%d').tolist()}"
    )

print(f"Fetch range: {available_weeks.min().date()} to "
      f"{available_weeks.max().date()}")
print(f"Rows: {len(weekly_panel):,}")
print(f"Stocks: {weekly_panel['Ticker'].nunique():,}")
print(f"Weeks: {weekly_panel['week_date'].nunique():,}")

display(weekly_panel.head())

In [ ]:
from pathlib import Path
import pandas as pd

cache = Path("sector_mapping.csv")

if cache.exists():
    sectors = pd.read_csv(cache)
else:
    url = (
        "https://raw.githubusercontent.com/datasets/"
        "s-and-p-500-companies/main/data/constituents.csv"
    )

    sectors = (
        pd.read_csv(url)[["Symbol", "GICS Sector", "GICS Sub-Industry"]]
        .rename(columns={
            "Symbol": "Ticker",
            "GICS Sector": "sector",
            "GICS Sub-Industry": "sub_industry",
        })
    )

    # Match Yahoo-style share-class tickers: BRK.B → BRK-B
    sectors["Ticker"] = (
        sectors["Ticker"].str.strip().str.upper()
        .str.replace(".", "-", regex=False)
    )
    sectors["mapping_download_date"] = (
        pd.Timestamp.now(tz="UTC").date().isoformat()
    )
    sectors.to_csv(cache, index=False)

display(sectors.head())

In [ ]:
# Replace weekly_panel with your actual DataFrame name.
panel = weekly_panel.copy()

panel["Ticker"] = (
    panel["Ticker"].str.strip().str.upper()
    .str.replace(".", "-", regex=False)
)

panel = panel.merge(
    sectors[["Ticker", "sector", "sub_industry"]],
    on="Ticker",
    how="left",
    validate="many_to_one",
)

missing = sorted(
    panel.loc[panel["sector"].isna(), "Ticker"].unique()
)
print(f"Unmapped tickers ({len(missing)}):", missing)

In [ ]:
sector_patch = {
    "AAL":  "Industrials",              # American Airlines
    "ARM":  "Information Technology",   # Arm Holdings
    "AVB":  "Real Estate",              # AvalonBay Communities
    "BBWI": "Consumer Discretionary",   # Bath & Body Works
    "BIO":  "Health Care",              # Bio-Rad Laboratories
    "BK":   "Financials",               # Bank of New York Mellon
    "BWA":  "Consumer Discretionary",   # BorgWarner
    "CAG":  "Consumer Staples",         # Conagra Brands
    "CE":   "Materials",                # Celanese
    "CPB":  "Consumer Staples",         # Campbell's
    "CTRA": "Energy",                   # Coterra Energy
    "CZR":  "Consumer Discretionary",   # Caesars Entertainment
    "DAY":  "Information Technology",   # Dayforce
    "EMN":  "Materials",                # Eastman Chemical
    "ENPH": "Information Technology",   # Enphase Energy
    "EPAM": "Information Technology",   # EPAM Systems
    "EQR":  "Real Estate",              # Equity Residential
    "ETSY": "Consumer Discretionary",   # Etsy
    "FMC":  "Materials",                # FMC Corporation
    "HOLX": "Health Care",              # Hologic
    "ILMN": "Health Care",              # Illumina
    "MHK":  "Consumer Discretionary",   # Mohawk Industries
    "MKTX": "Financials",               # MarketAxess
    "MOH":  "Health Care",              # Molina Healthcare
    "MSTR": "Information Technology",   # Strategy / MicroStrategy
    "MTCH": "Communication Services",   # Match Group
    "NET":  "Information Technology",   # Cloudflare
    "PAYC": "Information Technology",   # Paycom
    "POOL": "Consumer Discretionary",   # Pool Corporation
    "RHI":  "Industrials",              # Robert Half
    "RKT":  "Financials",               # Rocket Companies
    "SEE":  "Materials",                # Sealed Air
    "SPOT": "Communication Services",   # Spotify
    "VFC":  "Consumer Discretionary",   # VF Corporation
    "WHR":  "Consumer Discretionary",   # Whirlpool
    "WSO":  "Industrials",              # Watsco
    "ZS":   "Information Technology",   # Zscaler
}

# panel = your feature DataFrame AFTER joining the original sector mapping.
panel["Ticker"] = (
    panel["Ticker"].str.strip().str.upper()
    .str.replace(".", "-", regex=False)
)

# Fill missing sectors only; preserve existing assignments.
needs_sector = panel["sector"].isna()

panel.loc[needs_sector, "sector"] = (
    panel.loc[needs_sector, "Ticker"].map(sector_patch)
)

remaining = sorted(
    panel.loc[panel["sector"].isna(), "Ticker"].unique()
)

print("Still unmapped:", remaining)
assert not remaining, f"Resolve these tickers before neutralizing: {remaining}"

display(
    panel[["Ticker", "sector"]]
    .drop_duplicates()
    .sort_values("Ticker")
)

In [ ]:
completed_mapping = (
    panel[["Ticker", "sector"]]
    .drop_duplicates()
    .sort_values("Ticker")
)

assert not completed_mapping["Ticker"].duplicated().any()

completed_mapping.to_csv(
    "sector_mapping_completed.csv",
    index=False
)

In [ ]:
completed_mapping['sector'].unique()

In [ ]:
from cross_sectional_panel import build_cross_sectional_panel

# Score Jan 1 as the lookback, then filter it out only after lagging.
scored_with_lookback = build_cross_sectional_panel(panel)
panel_end_week = requested_weeks[-1]
cross_sectional_panel = (
    scored_with_lookback.loc[
        scored_with_lookback["week_date"].between(
            PANEL_START_WEEK, panel_end_week
        )
    ]
    .sort_values(["week_date", "Ticker"])
    .reset_index(drop=True)
)

assert cross_sectional_panel["week_date"].min() == PANEL_START_WEEK
assert cross_sectional_panel["week_date"].max() == panel_end_week
first_week = cross_sectional_panel["week_date"].eq(PANEL_START_WEEK)
assert cross_sectional_panel.loc[first_week, "prev_week_date"].eq(
    FETCH_START_WEEK
).all()
assert {"active_return_train", "active_return_fwd"}.issubset(
    cross_sectional_panel.columns
)

panel_output = Path(
    "data/gold/cross_sectional_panel_2025_jan_feb_v2.parquet"
)
panel_output.parent.mkdir(parents=True, exist_ok=True)
cross_sectional_panel.to_parquet(panel_output, index=False)

print(f"Panel range: {PANEL_START_WEEK.date()} to "
      f"{panel_end_week.date()}")
print(f"Rows: {len(cross_sectional_panel):,}")
print(f"Weeks: {cross_sectional_panel['week_date'].nunique()}")
print(f"Saved: {panel_output}")
display(cross_sectional_panel.head())

In [ ]:
cross_sectional_panel.describe()

## Build and publish the complete historical panel

This job reads every weekly Parquet snapshot from `features_parquet`, validates continuous Wednesday coverage, constructs the sector-neutral panel including `active_return_train` and `active_return_fwd`, and publishes one overwrite-protected `panel.parquet` per week plus `_manifest.json`. Reruns resume safely by skipping existing weekly objects; the target-bearing panel uses its own v2 destination prefix.

In [ ]:
from argparse import Namespace

from historical_panel_s3 import (
    DEFAULT_BUCKET,
    DEFAULT_DESTINATION_PREFIX,
    DEFAULT_LOCAL_OUTPUT,
    DEFAULT_SOURCE_PREFIX,
    run,
)

full_history_job = Namespace(
    bucket=DEFAULT_BUCKET,
    source_prefix=DEFAULT_SOURCE_PREFIX,
    destination_prefix=DEFAULT_DESTINATION_PREFIX,
    profile="admin",
    region="ap-southeast-2",
    sector_map=[
        "sector_mapping_completed.csv",
        "sector_mapping.csv",
    ],
    local_output=str(DEFAULT_LOCAL_OUTPUT),
    workers=12,
    upload=True,
)

historical_manifest = run(full_history_job)
historical_manifest["validation"]